# Enhanced GSMStateFn with State Function Type Intelligence

This notebook tests the enhanced GSMStateFn class that now includes:
- State function type awareness (U, F, H, G)
- Variable mapping intelligence
- Auto-detection of state function types
- Intelligent methods for thermodynamic properties
- Transformation target identification

The enhancements make the GSMStateFn class more intelligent and self-aware of its role in thermodynamic transformations.

In [2]:
# Setup and imports - enhanced version
import sys
import os

# Ensure we can import from the current directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else 'gsm_state_fn_02.ipynb'))
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print("Enhanced GSMStateFn Testing Notebook")
print("=" * 40)
print(f"Working directory: {current_dir}")

# Test basic imports first
print("\nTesting basic imports...")

try:
    import sympy as sp
    print("✓ SymPy imported successfully")
    sp_available = True
except ImportError as e:
    print(f"✗ SymPy import failed: {e}")
    sp_available = False

# Test our enhanced GSMStateFn imports
print("\nTesting enhanced GSMStateFn imports...")

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import StateFunction
    print("✓ StateFunction enum imported successfully")
    print(f"Available state functions: {[sf.value for sf in StateFunction]}")
    state_fn_enum_available = True
except ImportError as e:
    print(f"✗ StateFunction enum import failed: {e}")
    state_fn_enum_available = False

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import NATURAL_VARIABLES_MAPPING, TRANSFORMATION_MAPPING
    print("✓ Variable mappings imported successfully")
    mappings_available = True
except ImportError as e:
    print(f"✗ Variable mappings import failed: {e}")
    mappings_available = False

try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import GSMStateFn
    print("✓ Enhanced GSMStateFn imported successfully")
    gsm_state_fn_available = True
except ImportError as e:
    print(f"✗ GSMStateFn import failed: {e}")
    gsm_state_fn_available = False

# Test GSM variables
try:
    from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar
    print("✓ GSM variables imported successfully")
    gsm_vars_available = True
except ImportError:
    print("Note: GSM variables not available, will use standard sympy symbols")
    gsm_vars_available = False

print(f"\nImport Summary:")
print(f"  SymPy: {'✓' if sp_available else '✗'}")
print(f"  StateFunction enum: {'✓' if state_fn_enum_available else '✗'}")
print(f"  Variable mappings: {'✓' if mappings_available else '✗'}")
print(f"  GSMStateFn: {'✓' if gsm_state_fn_available else '✗'}")
print(f"  GSM variables: {'✓' if gsm_vars_available else '✗'}")

# Set global flags for other cells
globals()['sp_available'] = sp_available
globals()['gsm_state_fn_available'] = gsm_state_fn_available
globals()['gsm_vars_available'] = gsm_vars_available

print("\n✅ Setup completed!")

Enhanced GSMStateFn Testing Notebook
Working directory: /home/rch/Coding/bmcs_matmod/bmcs_matmod/gsm_lagrange/core2

Testing basic imports...
✓ SymPy imported successfully

Testing enhanced GSMStateFn imports...
✓ StateFunction enum imported successfully
Available state functions: ['U', 'F', 'H', 'G']
✓ Variable mappings imported successfully
✓ Enhanced GSMStateFn imported successfully
✓ GSM variables imported successfully

Import Summary:
  SymPy: ✓
  StateFunction enum: ✓
  Variable mappings: ✓
  GSMStateFn: ✓
  GSM variables: ✓

✅ Setup completed!


## 1. Testing State Function Type Intelligence

Let's test the enhanced GSMStateFn with explicit state function type specification.

In [3]:
# Create symbols for our test - using available imports
if gsm_vars_available:
    # Using gsm_vars Scalar
    T, S = Scalar('T'), Scalar('S')
    eps, sig = Scalar('eps'), Scalar('sig')
    omega, Y = Scalar('omega'), Scalar('Y')
    E = Scalar('E')
    symbol_type = "gsm_vars.Scalar"
else:
    # Using standard sympy symbols
    T, S = sp.Symbol('T'), sp.Symbol('S')
    eps, sig = sp.Symbol('eps'), sp.Symbol('sig')
    omega, Y = sp.Symbol('omega'), sp.Symbol('Y')
    E = sp.Symbol('E')
    symbol_type = "sympy.Symbol"

print("Created thermodynamic symbols:")
print(f"  Thermal: T={T}, S={S}")
print(f"  Mechanical: eps={eps}, sig={sig}")
print(f"  Internal: omega={omega}, Y={Y}")
print(f"  Parameter: E={E}")
print(f"  Symbol type: {symbol_type}")
print(f"  T is {type(T).__name__} from {type(T).__module__}")
print("\n✅ Symbols created successfully!")

Created thermodynamic symbols:
  Thermal: T=T, S=S
  Mechanical: eps=eps, sig=sig
  Internal: omega=omega, Y=Y
  Parameter: E=E
  Symbol type: gsm_vars.Scalar
  T is Scalar from bmcs_matmod.gsm_lagrange.core2.gsm_vars

✅ Symbols created successfully!


In [4]:
# Test 1: Create Helmholtz Free Energy F(T, ε, ω) with explicit type
print("Test 1: Creating Helmholtz Free Energy F(T, ε, ω) with explicit type")
print("=" * 70)

# Define Helmholtz free energy expression
F_expr = E * (1 - omega) * eps**2

# Create GSMStateFn with explicit Helmholtz type
F_state_fn = GSMStateFn(
    fn_expr=F_expr,
    th_x_var=T,      # Temperature (natural for Helmholtz)
    th_y_var=S,      # Entropy (conjugate)
    mc_x_var=eps,    # Strain (natural for Helmholtz)  
    mc_y_var=sig,    # Stress (conjugate)
    Eps_var=omega,   # Internal natural variable
    Sig_var=Y,       # Internal conjugate variable
    state_function_type=StateFunction.HELMHOLTZ
)

print(f"State Function Type: {F_state_fn.state_function_type}")
print(f"State Function Name: {F_state_fn.state_function_name}")
print(f"Expression: {F_state_fn.fn_expr}")
print(f"Representation: {F_state_fn}")
print()

Test 1: Creating Helmholtz Free Energy F(T, ε, ω) with explicit type
State Function Type: StateFunction.HELMHOLTZ
State Function Name: F
Expression: E*eps**2*(1 - omega)
Representation: GSMStateFn[F](f(T, eps, omega) = E*eps**2*(1 - omega))



## 2. Testing Intelligence Methods

The enhanced GSMStateFn has several intelligent methods that provide information about expected variable organization and thermodynamic properties.

In [5]:
# Test 2: Intelligence methods for Helmholtz function
print("Test 2: Intelligence Methods for Helmholtz F(T, ε, ω)")
print("=" * 55)

print("Variable Organization Intelligence:")
print(f"  Expected Natural Variables: {F_state_fn.get_expected_natural_variables()}")
print(f"  Expected Conjugate Variables: {F_state_fn.get_expected_conjugate_variables()}")
print()

print("Thermodynamic Properties:")
print(f"  Is Thermally Intensive (T natural): {F_state_fn.is_thermally_intensive()}")
print(f"  Is Mechanically Intensive (σ natural): {F_state_fn.is_mechanically_intensive()}")
print()

print("Transformation Intelligence:")
targets = F_state_fn.get_legendre_transformation_targets()
if targets:
    target_names = [t.value for t in targets]
    print(f"  Possible Transformation Targets: {target_names}")
else:
    print("  No transformation targets found")
print()

Test 2: Intelligence Methods for Helmholtz F(T, ε, ω)
Variable Organization Intelligence:
  Expected Natural Variables: ['T', 'eps', 'Eps']
  Expected Conjugate Variables: ['S', 'sig', 'Sig']

Thermodynamic Properties:
  Is Thermally Intensive (T natural): True
  Is Mechanically Intensive (σ natural): False

Transformation Intelligence:
  Possible Transformation Targets: ['U', 'G', 'H']



## 3. Testing Auto-Detection of State Function Types

The enhanced GSMStateFn can automatically detect the state function type based on variable patterns.

In [6]:
# Test 3: Auto-detection of state function type
print("Test 3: Auto-detection of State Function Type")
print("=" * 50)

# Create with auto-detection (T and eps suggest Helmholtz)
auto_F = GSMStateFn.create_with_auto_type_detection(
    fn_expr=F_expr,
    th_x_var=T,      # T suggests Helmholtz or Gibbs
    th_y_var=S,
    mc_x_var=eps,    # eps suggests Helmholtz or Internal Energy
    mc_y_var=sig,    # Combined: T + eps = Helmholtz
    Eps_var=omega,
    Sig_var=Y
)

print(f"Auto-detected type: {auto_F.state_function_type}")
print(f"Matches explicit type: {auto_F.state_function_type == F_state_fn.state_function_type}")
print(f"Auto-detected representation: {auto_F}")
print()

# Test different patterns
print("Testing different auto-detection patterns:")

# Internal Energy: S + eps
auto_U = GSMStateFn.create_with_auto_type_detection(
    fn_expr=F_expr + T*S,  # U = F + TS
    th_x_var=S,      # S suggests Internal Energy or Enthalpy
    th_y_var=T,
    mc_x_var=eps,    # eps suggests Internal Energy or Helmholtz
    mc_y_var=sig,    # Combined: S + eps = Internal Energy
    Eps_var=omega,
    Sig_var=Y
)
print(f"  S + eps pattern detected as: {auto_U.state_function_type}")

# Gibbs: T + sig
auto_G = GSMStateFn.create_with_auto_type_detection(
    fn_expr=F_expr - eps*sig,  # G = F - εσ
    th_x_var=T,      # T suggests Helmholtz or Gibbs
    th_y_var=S,
    mc_x_var=sig,    # sig suggests Gibbs or Enthalpy
    mc_y_var=eps,    # Combined: T + sig = Gibbs
    Eps_var=omega,
    Sig_var=Y
)
print(f"  T + sig pattern detected as: {auto_G.state_function_type}")

# Enthalpy: S + sig
auto_H = GSMStateFn.create_with_auto_type_detection(
    fn_expr=F_expr + T*S + eps*sig,  # H = F + TS + εσ
    th_x_var=S,      # S suggests Internal Energy or Enthalpy
    th_y_var=T,
    mc_x_var=sig,    # sig suggests Gibbs or Enthalpy
    mc_y_var=eps,    # Combined: S + sig = Enthalpy
    Eps_var=omega,
    Sig_var=Y
)
print(f"  S + sig pattern detected as: {auto_H.state_function_type}")
print()

Test 3: Auto-detection of State Function Type
Auto-detected type: StateFunction.HELMHOLTZ
Matches explicit type: True
Auto-detected representation: GSMStateFn[F](f(T, eps, omega) = E*eps**2*(1 - omega))

Testing different auto-detection patterns:
  S + eps pattern detected as: StateFunction.INTERNAL_ENERGY
  T + sig pattern detected as: StateFunction.GIBBS
  S + sig pattern detected as: StateFunction.ENTHALPY



## 5. Testing Enhanced Print Overview

The print_overview method shows comprehensive intelligence information for every state function.

In [7]:
# Test 5: Enhanced print overview
print("Test 5: Enhanced Print Overview")
print("=" * 35)
print("\nOverview for Helmholtz state function:")
print("-" * 50)
F_state_fn.print_overview()

print("Overview for Internal Energy state function:")
print("-" * 50)
auto_U.print_overview()

Test 5: Enhanced Print Overview

Overview for Helmholtz state function:
--------------------------------------------------
GSM State Function Overview
Type: F
Expression: E*eps**2*(1 - omega)

Natural Variables (independent):
  Thermal: T
  Mechanical: eps
  Internal: omega

Conjugate Variables (derivatives):
  Thermal: S
  Mechanical: sig
  Internal: Y

Expected Variable Organization:
  Natural: ['T', 'eps', 'Eps']
  Conjugate: ['S', 'sig', 'Sig']
  Thermally Intensive: True
  Mechanically Intensive: False
  Transformation Targets: ['U', 'G', 'H']

Overview for Internal Energy state function:
--------------------------------------------------
GSM State Function Overview
Type: U
Expression: E*eps**2*(1 - omega) + S*T

Natural Variables (independent):
  Thermal: S
  Mechanical: eps
  Internal: omega

Conjugate Variables (derivatives):
  Thermal: T
  Mechanical: sig
  Internal: Y

Expected Variable Organization:
  Natural: ['S', 'eps', 'Eps']
  Conjugate: ['T', 'sig', 'Sig']
  Thermally 

## 6. Testing Variable Substitution with Type Preservation

The substitute_variables method preserves the state function type in all cases.

In [8]:
# Test 6: Variable substitution preserving type
print("Test 6: Variable Substitution Preserving Type")
print("=" * 48)

# Perform substitution
substitutions = {E: 100, omega: 0.1}
F_substituted = F_state_fn.substitute_variables(substitutions)

print(f"Original type: {F_state_fn.state_function_type}")
print(f"Substituted type: {F_substituted.state_function_type}")
print(f"Type preserved: {F_substituted.state_function_type == F_state_fn.state_function_type}")
print()
print(f"Original expression: {F_state_fn.fn_expr}")
print(f"Substituted expression: {F_substituted.fn_expr}")
print()
print(f"Original: {F_state_fn}")
print(f"Substituted: {F_substituted}")
print()

Test 6: Variable Substitution Preserving Type
Original type: StateFunction.HELMHOLTZ
Substituted type: StateFunction.HELMHOLTZ
Type preserved: True

Original expression: E*eps**2*(1 - omega)
Substituted expression: 90.0*eps**2

Original: GSMStateFn[F](f(T, eps, omega) = E*eps**2*(1 - omega))
Substituted: GSMStateFn[F](f(T, eps, omega) = 90.0*eps**2)



## 7. Testing Constitutive Relations with Intelligence

Let's verify that constitutive relations work correctly with the intelligent state functions.

In [ ]:
# Test 7: Constitutive relations with intelligence
print("Test 7: Constitutive Relations with State Function Intelligence")
print("=" * 65)

# Compute constitutive relations for Helmholtz
F_relations = F_state_fn.compute_constitutive_relations()

print(f"Constitutive relations for {F_state_fn.state_function_type.value}(T,ε,ω):")
for var, expr in F_relations.items():
    print(f"  {var} = ∂F/∂{var.name if hasattr(var, 'name') else str(var)} = {expr}")
print()

# Test specific relation methods
thermal_rel = F_state_fn.get_thermal_constitutive_relation()
mechanical_rels = F_state_fn.get_mechanical_constitutive_relations()
internal_rels = F_state_fn.get_internal_constitutive_relations()

print("Specific constitutive relations:")
print(f"  Thermal: {thermal_rel[0]} = {thermal_rel[1]}")
print(f"  Mechanical: {mechanical_rels[0][0]} = {mechanical_rels[0][1]}")
print(f"  Internal: {internal_rels[0][0]} = {internal_rels[0][1]}")
print()

Test 7: Constitutive Relations with State Function Intelligence
Constitutive relations for F(T,ε,ω):
  S = ∂F/∂S = 0
  sig = ∂F/∂sig = 2*E*eps*(1 - omega)
  Y = ∂F/∂Y = -E*eps**2

Specific constitutive relations:
  Thermal: S = 0
  Mechanical: sig = 2*E*eps*(1 - omega)
  Internal: Y = -E*eps**2



## 8. Testing Variable Mappings and Transformation Data

Let's examine the variable mappings and transformation data that are now part of the GSMStateFn module.

In [10]:
# Test 8: Variable mappings and transformation data
print("Test 8: Variable Mappings and Transformation Data")
print("=" * 52)

print("Natural Variables Mapping:")
for state_fn, (natural, conjugate) in NATURAL_VARIABLES_MAPPING.items():
    print(f"  {state_fn.value}: natural={natural}, conjugate={conjugate}")
print()

print("Sample Transformation Mappings:")
transformation_count = 0
for (source, target), (thermal_coeff, work_coeff) in TRANSFORMATION_MAPPING.items():
    if transformation_count < 6:  # Show first 6 transformations
        coeffs_str = ""
        if thermal_coeff != 0:
            coeffs_str += f"{thermal_coeff:+d}*T*S "
        if work_coeff != 0:
            coeffs_str += f"{work_coeff:+d}*σ*ε"
        coeffs_str = coeffs_str.strip()
        
        if coeffs_str:
            formula = f"{target.value} = {source.value} {coeffs_str}"
        else:
            formula = f"{target.value} = {source.value}"
            
        print(f"  {source.value} → {target.value}: {formula}")
        transformation_count += 1

print(f"  ... and {len(TRANSFORMATION_MAPPING) - 6} more transformations")
print()

print("✓ Intelligence data successfully moved from GSMThermodynBox2 to GSMStateFn")
print()

Test 8: Variable Mappings and Transformation Data
Natural Variables Mapping:
  U: natural=['S', 'eps', 'Eps'], conjugate=['T', 'sig', 'Sig']
  F: natural=['T', 'eps', 'Eps'], conjugate=['S', 'sig', 'Sig']
  H: natural=['S', 'sig', 'Eps'], conjugate=['T', 'eps', 'Sig']
  G: natural=['T', 'sig', 'Eps'], conjugate=['S', 'eps', 'Sig']

Sample Transformation Mappings:
  U → F: F = U -1*T*S
  F → U: U = F +1*T*S
  U → H: H = U +1*σ*ε
  H → U: U = H -1*σ*ε
  F → G: G = F -1*σ*ε
  G → F: F = G +1*σ*ε
  ... and 6 more transformations

✓ Intelligence data successfully moved from GSMThermodynBox2 to GSMStateFn



## 9. Summary and Next Steps

Summary of enhancements made to GSMStateFn:

In [11]:
# Test 9: Summary of GSMStateFn Enhancements
print("Test 9: Summary of GSMStateFn Enhancements")
print("=" * 44)
print("✅ Enhancements Successfully Implemented:")
print("  1. State Function Type Awareness (StateFunction enum)")
print("  2. Variable Mapping Intelligence (NATURAL_VARIABLES_MAPPING)")
print("  3. Transformation Data (TRANSFORMATION_MAPPING)")
print("  4. Auto-detection of State Function Types")
print("  5. Intelligent Methods:")
print("     - get_expected_natural_variables()")
print("     - get_expected_conjugate_variables()")
print("     - is_thermally_intensive()")
print("     - is_mechanically_intensive()")
print("     - get_legendre_transformation_targets()")
print("  6. Enhanced String Representations")
print("  7. Type-preserving Variable Substitution")
print("  8. Enhanced Print Overview with Intelligence")
print("  9. Mandatory State Function Type (No Optional handling)")
print()
print("🎯 Ready for Next Steps:")
print("  - Create convenience subclasses (InternalEnergy, Helmholtz, Enthalpy, Gibbs)")
print("  - Implement simplified constructors like F(expression, eps_var, sig_var, ...)")
print("  - Integration with GSMThermodynBox2 for seamless transformations")
print("  - Enhanced validation and consistency checking")
print()
print("✨ GSMStateFn is now significantly more intelligent and requires explicit typing!")
print()

Test 9: Summary of GSMStateFn Enhancements
✅ Enhancements Successfully Implemented:
  1. State Function Type Awareness (StateFunction enum)
  2. Variable Mapping Intelligence (NATURAL_VARIABLES_MAPPING)
  3. Transformation Data (TRANSFORMATION_MAPPING)
  4. Auto-detection of State Function Types
  5. Intelligent Methods:
     - get_expected_natural_variables()
     - get_expected_conjugate_variables()
     - is_thermally_intensive()
     - is_mechanically_intensive()
     - get_legendre_transformation_targets()
  6. Enhanced String Representations
  7. Type-preserving Variable Substitution
  8. Enhanced Print Overview with Intelligence
  9. Mandatory State Function Type (No Optional handling)

🎯 Ready for Next Steps:
  - Create convenience subclasses (InternalEnergy, Helmholtz, Enthalpy, Gibbs)
  - Implement simplified constructors like F(expression, eps_var, sig_var, ...)
  - Integration with GSMThermodynBox2 for seamless transformations
  - Enhanced validation and consistency checkin